In [242]:
import pandas as pd
from transformers import AutoTokenizer, BertTokenizer
from nltk import word_tokenize
from datasets import load_dataset

ds = load_dataset("../data/wikitext-2/")

In [243]:
import re, unicodedata

def is_valid(row):
    text = row["text"].strip()
    if text == "":
        return False
    if re.match(r"^=+.*=+$", text):
        return False
    return True

def clean_text(row):
    t = row["text"]
    # lowercase + unicode → ascii
    t = unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode("ascii")
    # fix wikitext artifacts
    t = re.sub(r"@-@", "-", t)
    t = re.sub(r"@\.@", ".", t)
    t = re.sub(r"@,@", ",", t)
    t = re.sub(r"[–—−]", "-", t)
    # remove parentheses and content
    t = re.sub(r"\([^)]*\)", " ", t)
    # roman numerals i–v → digits
    t = re.sub(r"\b(i|ii|iii|iv|v)\b",lambda m: str({"i": 1, "ii": 2, "iii": 3, "iv": 4, "v": 5}[m.group()]),t,)
    # normalize ellipses (3+ dots) → "."
    t = re.sub(r"\.{3,}", ".", t)
    # keep: . , ; : ! ? ' - and spaces
    t = re.sub(r"[^a-z0-9\.\,\;\:\!\?\'\-\s]", " ", t)
    # collapse patterns like "6 , 000" or "3 , 000" → "<num>"
    t = re.sub(r"\b\d+\s*,\s*\d+\b", "<num>", t)
    # (optional) handle ordinals like 19th, 21st → "19 th"
    t = re.sub(r"\b(\d+)(st|nd|rd|th)\b", r"\1 th", t)
    # number bucketing
    def num_bucket(m):
        n = int(m.group())
        if 0 <= n <= 9:
            return str(n)
        if 10 <= n < 100:
            return "<2dnum>"
        if 1000 <= n <= 2099:
            return "<year>"
        return "<num>"
    t = re.sub(r"\d+", num_bucket, t)
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return {"clean_text": t}

In [244]:
ds = ds.filter(is_valid)
ds = ds.map(clean_text)

Map:   0%|          | 0/17556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1841 [00:00<?, ? examples/s]

Map:   0%|          | 0/2185 [00:00<?, ? examples/s]

In [245]:
corpus = " ".join(ds["train"]["clean_text"])
print(sorted(set(corpus)))
print(len(corpus.split()))
print(len(set(corpus.split())))

[' ', '!', "'", ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
1893749
61166


In [246]:
import tokenizers
special_tokens = ["<pad>", "<unk>", "<eos>", "<num>", "<year>", "<2dnum>"]
vocab_size = 25_000
tokenizer = tokenizers.Tokenizer(tokenizers.models.WordLevel(unk_token="<unk>"))
tokenizer.add_special_tokens(special_tokens)
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
tokenizer.normalizer = tokenizers.normalizers.Lowercase()
tokenizer.train_from_iterator(
    ds["train"]["clean_text"], tokenizers.trainers.WordLevelTrainer(vocab_size=vocab_size, special_tokens=special_tokens)
)

In [251]:
i = 15127
print(ds["train"]["text"][i])
text = ds["train"]["clean_text"][i]
print(text, len(text.split()))
print("="*30)
tokens = tokenizer.encode(text).tokens
print(tokens, len(tokens))

 The Rocky Mountains of southwestern Montana at the headwaters of the Missouri River first rose in the Laramide Orogeny , a mountain @-@ building episode that occurred from around 70 to 45 million years ago ( the end of the Mesozoic through the early Cenozoic ) . This orogeny uplifted Cretaceous rocks along the western side of the Western Interior Seaway , a vast shallow sea that stretched from the Arctic Ocean to the Gulf of Mexico , and deposited the sediments that now underlie much of the drainage basin of the Missouri River . This Laramide uplift caused the sea to retreat and laid the framework for a vast drainage system of rivers flowing from the Rocky and Appalachian Mountains , the predecessor of the modern @-@ day Mississippi watershed . The Laramide Orogeny is essential to modern Missouri River hydrology , as snow and ice melt from the Rockies provide the majority of the flow in the Missouri and its tributaries . 

the rocky mountains of southwestern montana at the headwaters 